# 25 -- Striatum centroid audit in the RAS-resampled frame (2026-09-11)

A second Opus review of `notebooks/24_slab2d_architecture_diversity.ipynb`'s
negative gate result found the real root cause was not the geometry-
rounding bug fixed earlier that day, but that `config.CROP_CENTER_MM` is a
**population-median** offset -- the median row of
`notebooks/01_eda_volumes.ipynb` section 6a's per-volume striatum-centroid-
offset table, whose z component has **sd ~32.5mm** (5-95th percentile
range roughly -97mm to +4mm) in that notebook's measurement. A 12mm-thick
slab fixed at that one point contains real striatal tissue for only
~14-47% of subjects by a normal approximation; the 3D track's 108mm-thick
crop tolerates the same offset error by sheer thickness (~89% by the same
approximation) -- explaining why `slab2d` failed but the 3D CNN (0.4520
solo log loss) clearly works.

That estimate has two real caveats worth resolving before trusting it:
1. Section 6a computed offsets on **raw array axes** (`img.get_fdata()`,
   no resampling), while `crop_or_pad` applies `CROP_CENTER_MM` in the
   **RAS-resampled** frame `resample_to_spacing` produces. For the 40% of
   volumes with oblique affines (README.md), those are different
   coordinate frames -- raw-axis sd may overstate the true anatomical
   spread.
2. Section 6a sampled only 119 of the 1362 training volumes.

This notebook redoes the measurement correctly: `features.striatum_mask`
applied to the **RAS-resampled** volume (the exact frame `crop_or_pad`
uses, the same resampling `load_volume` itself performs) for **all 1362**
volumes, reporting only aggregate statistics -- never per-row/per-uid
values, per the AI-assistant data rule.

**Two things this answers**:
- Is the sd really ~32.5mm in the correct frame, or was the raw-axis
  measurement inflated by oblique-affine frame mismatch?
- **Higher-value question**: does the *production* 3D crop
  (`config.TARGET_SHAPE`/`CROP_SIZE_MM`/`CROP_CENTER_MM`, the crop the
  shipped 0.4185 recipe's CNN actually trains and scores on) fully contain
  the striatum for every subject, or does a meaningful fraction have the
  striatum at or beyond its edge? If the latter, per-subject centering is
  a genuine, previously-unidentified discrimination lever on the
  **production** model -- a bigger prize than `slab2d` ever was.

**Data handling**: loads real `.nii.gz` volumes (resampled, never cropped
to a fixed shape here -- the whole point is measuring where the striatum
actually sits before any fixed-crop assumption is applied) -- per the
AI-assistant data rule, this is **[RUN ME]**: run it yourself, share back
only the printed aggregate numbers. CPU-only, no GPU. Expect roughly the
same per-volume cost as building the 3D `volume_cache` from scratch
(~1.7s/volume per the project's own smoke-test benchmark) since the
dominant cost (`resample_to_spacing`) is the same operation -- so budget
~35-45 minutes for all 1362 volumes, no caching (this is a one-off
measurement, not reused by training).

In [ ]:
# [RUN ME] -- loads and resamples real volumes (no crop_or_pad -- this
# measures where the striatum actually sits, not where a fixed crop
# assumes it sits). CPU-only. ~35-45 min for all 1362 volumes.
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import nibabel as nib
import numpy as np
import pandas as pd

import config
import data
import features

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
uids = labels_df[config.UID_COLUMN].tolist()

offsets_mm = []  # (x, y, z) centroid offset from geometric center, mm
bbox_min_mm = []  # per-axis min of the striatum mask's bbox, mm offset
bbox_max_mm = []  # per-axis max of the striatum mask's bbox, mm offset
n_degenerate = 0

start = time.time()
for i, uid in enumerate(uids):
    path = config.NIFTI_DIR / f"{uid}.nii.gz"
    img = nib.load(str(path))
    resampled, _ = data.resample_to_spacing(img.get_fdata(), img.affine, config.TARGET_SPACING)

    mask = features.striatum_mask(resampled, config.TARGET_SPACING)
    if mask is None or not mask.any():
        n_degenerate += 1
        continue

    idx = np.argwhere(mask)
    shape = np.asarray(resampled.shape, dtype=float)
    center_vox = (shape - 1) / 2.0
    spacing = np.asarray(config.TARGET_SPACING)

    centroid_vox = idx.mean(axis=0)
    offsets_mm.append((centroid_vox - center_vox) * spacing)
    bbox_min_mm.append((idx.min(axis=0) - center_vox) * spacing)
    bbox_max_mm.append((idx.max(axis=0) - center_vox) * spacing)

    if (i + 1) % 200 == 0:
        elapsed = time.time() - start
        print(f"  {i + 1}/{len(uids)} processed, {elapsed:.0f}s elapsed "
              f"({elapsed / (i + 1):.2f}s/volume, ETA {elapsed / (i + 1) * (len(uids) - i - 1):.0f}s)")

offsets_mm = np.array(offsets_mm)
bbox_min_mm = np.array(bbox_min_mm)
bbox_max_mm = np.array(bbox_max_mm)
print(f"\ndone in {time.time() - start:.0f}s. "
      f"{len(offsets_mm)}/{len(uids)} volumes had a usable striatum mask "
      f"({n_degenerate} degenerate/skipped -- for comparison, "
      f"features.py/README.md report 0/1362 skipped for the classical "
      f"baseline's own use of this same mask, so a nonzero count here is "
      f"itself worth a second look before trusting the rest)")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# Answers question 1: is the RAS-resampled-frame sd close to section 6a's
# raw-axis sd (~32.5mm on z, 119-volume sample), or does it differ enough
# to matter -- i.e. was the raw-axis measurement frame-inflated?
axis_names = ["x (L-R)", "y (A-P)", "z (S-I)"]
for a, name in enumerate(axis_names):
    vals = offsets_mm[:, a]
    pct = np.percentile(vals, [5, 50, 95])
    print(f"offset_{name}: mean={vals.mean():+.1f}mm  sd={vals.std(ddof=1):.1f}mm  "
          f"5/50/95th pct = {pct[0]:+.1f} / {pct[1]:+.1f} / {pct[2]:+.1f} mm  "
          f"(n={len(vals)})")

print(f"\nconfig.CROP_CENTER_MM = {config.CROP_CENTER_MM} "
      "(should be close to each axis's median above, by construction -- "
      "notebooks/01 section 6a picked the median as the fixed offset)")
print("for comparison -- notebooks/01 section 6a (raw-axis frame, n=119): "
      "z sd~32.5mm, 5/50/95th pct ~ -96.9 / -15.5 / +4.0 mm")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# Answers question 2 (the higher-value one): does the PRODUCTION 3D crop
# (config.TARGET_SHAPE / CROP_SIZE_MM / CROP_CENTER_MM -- the crop the
# shipped 0.4185 recipe's CNN actually trains and scores on) fully contain
# the striatum for every subject?
half_size = np.asarray(config.CROP_SIZE_MM) / 2.0
crop_lo = np.asarray(config.CROP_CENTER_MM) - half_size
crop_hi = np.asarray(config.CROP_CENTER_MM) + half_size
print(f"production 3D crop window (mm offset from geometric center): "
      f"{list(zip(crop_lo.round(1), crop_hi.round(1)))}")

centroid_inside = np.all((offsets_mm >= crop_lo) & (offsets_mm <= crop_hi), axis=1)
bbox_fully_inside = np.all((bbox_min_mm >= crop_lo) & (bbox_max_mm <= crop_hi), axis=1)
bbox_any_outside_per_axis = [
    float(np.mean((bbox_min_mm[:, a] < crop_lo[a]) | (bbox_max_mm[:, a] > crop_hi[a])))
    for a in range(3)
]

print(f"\nstriatum CENTROID inside the production crop: "
      f"{centroid_inside.sum()}/{len(centroid_inside)} ({centroid_inside.mean():.1%})")
print(f"striatum full BBOX inside the production crop: "
      f"{bbox_fully_inside.sum()}/{len(bbox_fully_inside)} ({bbox_fully_inside.mean():.1%})")
print(f"fraction with bbox extending outside the crop, per axis: "
      f"x={bbox_any_outside_per_axis[0]:.1%}  y={bbox_any_outside_per_axis[1]:.1%}  "
      f"z={bbox_any_outside_per_axis[2]:.1%}")
print("\nfor context -- the shipped CNN alone (rung 3, README.md) scores "
      "log loss 0.4520 despite whatever coverage gap this shows, so this "
      "does not mean the production model is broken -- it quantifies "
      "*how much headroom* a per-subject-centered crop could plausibly "
      "recover, not a bug.")

**What we're looking for:** (1) whether the RAS-resampled-frame striatum-
offset spread is close to `notebooks/01` section 6a's raw-axis estimate
(sd~32.5mm on z, n=119) or meaningfully different (oblique-affine frame
mismatch was a real, unresolved caveat on that earlier number); (2) the
higher-value question -- does the *production* 3D crop actually contain
the striatum for (nearly) every subject, or is there a real, previously-
unmeasured coverage gap that per-subject centering could close?

**What we found:** *(paste: op03's per-axis mean/sd/percentiles vs.
section 6a's raw-axis numbers; op04's centroid-inside-crop and
bbox-fully-inside-crop percentages, and the per-axis outside fractions;
the degenerate-mask count from op02)*

**Decision / next step:** *(if the production crop's coverage is
essentially universal (fully-inside-crop close to 100%), the offset
spread doesn't threaten the shipped 0.4185 recipe and this audit is a
clean "no action needed" close-out of the second Opus review's open
question -- roadmap item 6 (architecture diversity) stays closed as a
negative-but-inconclusive result (see notebook 24 and RESOURCES.md), and
per-subject slab centering is not worth attempting (Opus's step 3, ~4h)
since the underlying offset problem it would fix barely exists in
practice.

If a meaningful fraction of subjects have the striatum bbox extending
outside the production crop (e.g. double-digit percent on any axis),
that's a genuine, newly-identified discrimination lever on the shipped
model itself -- bigger than anything explored today. Next step would be
to CV-test per-subject-centered cropping (via `features.striatum_mask`,
falling back to the current fixed `CROP_CENTER_MM` when the mask returns
`None`) as a candidate change to `data.load_volume`'s own crop, gated the
same way as every other change today (`evaluate.paired_gate`,
`min_effect=0.003`) -- a substantially bigger undertaking than slab2d
since it touches the production model's own preprocessing, and would need
the same TDD + local/platform smoke-test discipline as any other
production change, with real submission risk once/if it clears CV.)*